# 🌌 Discrete State Engine: Relativistic Quantum Walk Simulation
**Author:** [Aditya Kumar] | **Framework:** Numba (LLVM), Python, NumPy

### Abstract
This notebook demonstrates a high-performance, strictly decoupled Discrete State Engine. It simulates a multi-particle relativistic quantum walk across a dynamically generated, non-Euclidean topology.

Rather than relying on standard Python interpreted loops, this engine utilizes **Just-In-Time (JIT) LLVM compilation** and **O(1) Spatial Hash Grids** to achieve sub-millisecond physics calculations across hundreds of thousands of discrete spatial edges.

### Architecture Highlights
* **Decoupled Component Managers:** Topology, Fields, and Generators operate entirely independently.
* **C-Compiled Execution:** The core mathematical loops run in pure machine code via `@njit(fastmath=True)`.
* **Spatial Memory Optimization:** Eliminates O(N²) graph discovery bottlenecks using dense 3D Boolean arrays for instant neighbor lookups.

### ⚛️ The Physics: Relativistic Quantum Walks on Discrete Topologies

This engine simulates the evolution and collapse of quantum states in a bounded, discrete space. Unlike classical random walks where particles move to adjacent nodes with defined probabilities, this simulation models **Quantum Walks** where particles exist in a superposition of states, governed by relativistic momentum and probability amplitudes.

#### 1. The Discrete Spatial Tensor (Non-Euclidean Topology)
Continuous space is quantized into a discrete graph structure (e.g., a 6-fold hexagonal symmetry). The topology is bounded (Radius $R$), meaning the spatial boundaries reflect or constrain the wave functions, creating complex interference patterns.
* Let the graph be defined by vertices $V$ and edges $E$. The position of a particle is a discrete coordinate state $|q, r\rangle$.
* The topology generator maps this N-dimensional non-Euclidean graph into a flat, 1D continuous memory array to ensure $O(1)$ spatial hashing during execution.

#### 2. Relativistic Wave Evolution
Instead of a single scalar position, a particle's state is defined by its position and its momentum direction (a relativistic spinor analog).
* The state vector is $|\psi(t)\rangle = \sum_{x, d} \alpha_{x,d}(t) |x, d\rangle$, where $x$ is the spatial coordinate, $d$ is the momentum direction, and $\alpha$ is the complex probability amplitude.
* During each step, the engine applies a **Unitary Evolution Operator** ($\hat{U}$). The transition function governs how the wave function spreads to neighboring states based on momentum conservation and interference:
  $$|\psi(t+1)\rangle = \hat{U}|\psi(t)\rangle$$
* Because the transitions allow amplitudes to be complex numbers ($\mathbb{C}$), expanding wave fronts can destructively or constructively interfere, mimicking real quantum mechanical behavior.

#### 3. The Observer Effect (Wave Function Collapse)
After a defined number of discrete steps ($\Delta t$), the engine simulates a measurement phase. The expanded probability wave function must collapse into a single, definitive classical state.
* The probability of finding the particle at a specific state $|x\rangle$ is dictated by the Born Rule, taking the squared magnitude of the complex field amplitude:
  $$P(x) = |\psi(x)|^2 = \text{Re}(\alpha)^2 + \text{Im}(\alpha)^2$$
* The `sequential_relativistic_collapse` kernel acts as the observer. It performs a probabilistic weighted selection across the generated quantum field to permanently pin the particle to a new discrete coordinate.

#### 4. Multi-Particle Dynamics & Momentum Currents
The simulation processes $N$ particles simultaneously. As the wave functions expand (visualized as momentum currents mapping the vector field of the probabilities), particles collapse sequentially, ensuring that claimed classical states are registered and respected within the bounded tensor.

In [ ]:
# @title 🛠️ Environment Setup & Deep Path Injection
import os
import sys
import importlib

# 1. Clone/Update Repo
!rm -rf /content/discrete-state-engine
!git clone https://github.com/ckCrimson/discrete-state-engine.git /content/discrete-state-engine

# 2. Install FFMPEG (quietly)
!apt-get update -qq && apt-get install -y -qq ffmpeg

# 3. Search and Destroy: Find exactly where the folder landed
package_name = "particle_grid_simulator"
actual_parent_dir = None

for root, dirs, files in os.walk("/content"):
    if package_name in dirs:
        actual_parent_dir = root
        break

if actual_parent_dir:
    print(f"\n[*] Found '{package_name}' sitting in: {actual_parent_dir}")

    # 4. Inject the exact path into Python's brain
    if actual_parent_dir not in sys.path:
        sys.path.insert(0, actual_parent_dir)

    # 5. Force Colab to forget its old failed import attempts
    importlib.invalidate_caches()

    # 6. Import
    try:
        from particle_grid_simulator.src.generator.component_manager.component_manager import GeneratorComponentManager
        print("Framework successfully imported! You are good to go.")
    except ImportError as e:
        print(f"❌ Import failed: {e}")
        print("\n[!] CRITICAL: Colab's memory is totally jammed.")
        print("Go to the top menu: Runtime -> Restart runtime. Then run this cell again.")
else:
    print(f"\n❌ Could not find '{package_name}' anywhere.")
    !ls -la /content/discrete-state-engine

### 🧠 The Math & Memory Architecture
This block defines the core rules of the universe. Because Python object creation (like tuples and lists) is too slow for high-performance simulation, we push all logic down to the C-level.

1. **Topology Generation:** `dynamic_topology` calculates the neighboring coordinates for any given state based on a defined symmetry (e.g., 6-fold hexagonal).
2. **Transition Dynamics:** `dynamic_transition` governs the quantum momentum field, calculating the complex probability amplitudes of a particle moving between discrete states.
3. **Relativistic Collapse:** `sequential_relativistic_collapse` simulates the observer effect.

**Performance Note:** To prevent memory blowouts during Breadth-First Search (BFS) discovery, the collapse function utilizes a pre-allocated 3D Spatial Hash Grid (`visited_grid`). This reduces duplicate-checking from an O(N²) linear scan to an instantaneous O(1) memory lookup.

In [ ]:
# @title ⚙️ 1. Imports & Configuration
import os
import time
import numpy as np
from matplotlib import pyplot as plt
from matplotlib.animation import FuncAnimation, FFMpegWriter, PillowWriter
import matplotlib.colors as mcolors
import numba as nb

# FDS DOMAIN & MANAGER IMPORTS
from particle_grid_simulator.src.state.domain import State
from particle_grid_simulator.src.field.domain.data.field_algebra import FieldAlgebra
from particle_grid_simulator.src.field.domain.data.field_mapper import FieldMapper
from particle_grid_simulator.src.topology.domain.topology_domain import Topology
from particle_grid_simulator.src.generator.domain.data.generic_markovian_field_generator import GenericMarkovianFieldGeneratorData
from particle_grid_simulator.src.field.component_manager.component_manager import FieldComponentManager
from particle_grid_simulator.src.field.interfaces.storage import FieldKernelDataContract
from particle_grid_simulator.src.field.kernel.numba.storage.complex_field_storage_v2 import NumbaComplexFieldKernelStorage
from particle_grid_simulator.src.field.kernel.numba.translator.translator_v1 import NumbaFieldTranslator
from particle_grid_simulator.src.field.kernel.numba.utility.complex_field_utility_v2 import NumbaComplexUtility
from particle_grid_simulator.src.topology.component_manager.component_manager import TopologyComponentManager
from particle_grid_simulator.src.topology.kernel.numba.storage.storage_v1 import NumbaTopologyStorage, TopologyKernelDataContract
from particle_grid_simulator.src.topology.kernel.numba.translator.translator_v1 import NumbaTopologyTranslator
from particle_grid_simulator.src.topology.kernel.numba.utility.utility_v1 import NumbaTopologyUtility
from particle_grid_simulator.src.generator.component_manager.component_manager import GeneratorComponentManager
from particle_grid_simulator.src.generator.kernel.numba.storage.complex_field_storage_v2 import NumbaComplexCSRGeneratorStorage
from particle_grid_simulator.src.generator.kernel.numba.translator.generic_translator_v2 import GenericGeneratorTranslator
from particle_grid_simulator.src.generator.kernel.numba.utility.generic_utility_v2 import GenericGeneratorKernelUtility
from particle_grid_simulator.src.generator.iterfaces.storage import GeneratorKernelDataContract

# SYSTEM CONFIGURATION
NUM_PARTICLES = 10 # @param {type:"slider", min:1, max:50, step:1}
STEPS = 9 # @param {type:"slider", min:2, max:10, step:1}
ITERATIONS = 15  # @param {type:"integer"}
BOX_RADIUS = 100  # @param {type:"integer"}
PATH = "/content/plots" # Adjusted to save directly in Colab output

DELTA_POS = np.array([
    [1.0, 0.0], [-1.0, 0.0], [0.0, 1.0], [0.0, -1.0],
    [-1.0, 1.0], [1.0, -1.0]
], dtype=np.float64)

In [ ]:
# @title 🧠 2. Numba Physics Kernels & Topology

def build_relativistic_kernels(deltas: np.ndarray):
    n_neighbors = len(deltas)
    norm_scalar = 1.0 / np.sqrt(n_neighbors)

    @nb.njit(fastmath=True)
    def dynamic_topology(state_vec: np.ndarray) -> np.ndarray:
        q, r = state_vec[0], state_vec[1]
        out = np.empty((n_neighbors, 3), dtype=np.float64)
        for d_out in range(n_neighbors):
            out[d_out, 0] = q + deltas[d_out, 0]
            out[d_out, 1] = r + deltas[d_out, 1]
            out[d_out, 2] = float(d_out)
        return out

    @nb.njit(fastmath=True)
    def dynamic_transition(s_j: np.ndarray, s_i: np.ndarray) -> np.ndarray:
        d_in = s_j[2]
        d_out = s_i[2]
        if d_in == d_out:
            val = (2.0 / n_neighbors) - 1.0
        else:
            val = (2.0 / n_neighbors)
        return np.array([val + 0j], dtype=np.complex128)

    return dynamic_topology, dynamic_transition

# ==========================================
# THE O(1) SCALING FIX FOR COLLAPSE
# ==========================================
@nb.njit(fastmath=True)
def build_global_probability_grid(global_states: np.ndarray, global_fields: np.ndarray) -> np.ndarray:
    """Compresses the massive global quantum field into an instant O(1) lookup grid."""
    OFFSET = 300
    GRID_SIZE = 601
    prob_grid = np.zeros((GRID_SIZE, GRID_SIZE, 6), dtype=np.float64)

    for i in range(len(global_states)):
        g_state = global_states[i]
        amp = global_fields[i, 0]
        prob = amp.real**2 + amp.imag**2

        # Instant memory mapping
        q_idx = int(g_state[0]) + OFFSET
        r_idx = int(g_state[1]) + OFFSET
        d_idx = int(g_state[2])

        prob_grid[q_idx, r_idx, d_idx] = prob

    return prob_grid

@nb.njit(fastmath=True)
def sequential_relativistic_collapse_fast(
        p_state: np.ndarray, global_prob_grid: np.ndarray,
        claimed_grid: np.ndarray, deltas: np.ndarray, steps: int
) -> np.ndarray:
    """Particle only searches its local area, pulling probabilities instantly."""
    n_neighbors = len(deltas)
    max_bfs_nodes = ((2 * steps + 1) ** 2) * n_neighbors + 100

    frontier = np.empty((max_bfs_nodes, 3), dtype=np.float64)
    frontier[0] = p_state
    read_idx = 0
    write_idx = 1

    OFFSET = 300

    # 1. O(1) Grid for instant duplicate checking during local BFS
    visited_grid = np.zeros((601, 601, n_neighbors), dtype=np.bool_)
    visited_grid[int(p_state[0]) + OFFSET, int(p_state[1]) + OFFSET, int(p_state[2])] = True

    for _ in range(steps):
        layer_end = write_idx
        while read_idx < layer_end:
            curr = frontier[read_idx]
            read_idx += 1
            for d in range(n_neighbors):
                nq, nr = curr[0] + deltas[d, 0], curr[1] + deltas[d, 1]
                nd = float(d)

                q_idx = int(nq) + OFFSET
                r_idx = int(nr) + OFFSET
                d_idx = int(nd)

                if not visited_grid[q_idx, r_idx, d_idx]:
                    visited_grid[q_idx, r_idx, d_idx] = True
                    frontier[write_idx, 0] = nq
                    frontier[write_idx, 1] = nr
                    frontier[write_idx, 2] = nd
                    write_idx += 1

    # 2. Extract Valid Probabilities WITHOUT looping the global universe
    valid_probs = np.zeros(write_idx, dtype=np.float64)
    total_prob = 0.0

    for f_idx in range(write_idx):
        f_state = frontier[f_idx]
        q_idx = int(f_state[0]) + OFFSET
        r_idx = int(f_state[1]) + OFFSET
        d_idx = int(f_state[2])

        # O(1) Check: Is this spatial coordinate claimed by another particle?
        if not claimed_grid[q_idx, r_idx]:
            # O(1) Check: What is the global probability here?
            prob = global_prob_grid[q_idx, r_idx, d_idx]
            valid_probs[f_idx] = prob
            total_prob += prob

    # 3. Collapse via Born Rule
    if total_prob > 1e-15:
        r = np.random.random() * total_prob
        acc = 0.0
        for f_idx in range(write_idx):
            acc += valid_probs[f_idx]
            if r <= acc:
                return frontier[f_idx]

    return p_state

In [ ]:
# @title 🎬 3. Core Engine & Visualization Functions

def animate_relativistic_walk(history_wave, history_particles, deltas, steps, iterations, path):
    fig, ax = plt.subplots(figsize=(10, 10))
    all_states = []
    for w_states, _ in history_wave:
        if len(w_states) > 0:
            all_states.append(w_states)

    all_coords = np.concatenate(all_states)
    x_min, x_max = all_coords[:, 0].min() - 2, all_coords[:, 0].max() + 2
    y_min, y_max = all_coords[:, 1].min() - 2, all_coords[:, 1].max() + 2

    def update(frame):
        ax.clear()
        ax.set_facecolor('#050510')
        ax.set_xlim(x_min, x_max)
        ax.set_ylim(y_min, y_max)
        ax.set_aspect('equal')

        iter_idx = frame // (steps + 1)
        step_idx = frame % (steps + 1)

        if step_idx < steps:
            w_states, w_masses = history_wave[frame]
            if len(w_states) > 0:
                q = w_states[:, 0]
                r = w_states[:, 1]
                d_in = w_states[:, 2].astype(int)
                U = deltas[d_in, 0]
                V = deltas[d_in, 1]
                ax.quiver(q, r, U, V, w_masses, cmap='inferno', pivot='mid',
                          norm=mcolors.LogNorm(vmin=1e-5, vmax=1.0),
                          scale=35, width=0.003, headwidth=4, alpha=0.9)
            ax.set_title(f"Quantum Momentum Currents | Iteration {iter_idx + 1} | Step {step_idx + 1}", color='white')
        else:
            ax.set_title(f"Relativistic Collapse | Iteration {iter_idx + 1}", color='cyan', fontweight='bold')

        p_states = history_particles[frame]
        p_states_history = np.array(history_particles[:frame + 1])
        colors = ['#00F0FF', '#FF007F', '#7FFF00', '#FFD700', '#FF4500',
                  '#9400D3', '#00FF7F', '#DC143C', '#1E90FF', '#FF1493']

        for i in range(len(p_states)):
            c = colors[i % len(colors)]
            ax.plot(p_states_history[:, i, 0], p_states_history[:, i, 1],
                    color=c, alpha=0.5, linewidth=2.5, linestyle=':', zorder=9)
            ax.scatter(p_states[i, 0], p_states[i, 1], color=c,
                       s=120, edgecolors='white', linewidth=1.5, zorder=10)

    total_frames = iterations * (steps + 1)
    anim = FuncAnimation(fig, update, frames=total_frames, blit=False)

    os.makedirs(path, exist_ok=True)
    mp4_path = os.path.join(path, "relativistic_field_analysis.mp4")

    print(f"    🎬 Rendering {total_frames} frames...")
    writer = FFMpegWriter(fps=10)
    anim.save(mp4_path, writer=writer)
    print(f"    ✅ Analysis video ready: {mp4_path}")
    plt.close(fig) # Prevent duplicate display in Colab

# @title ⚡ 3. Core Engine Execution
def run_relativistic_multi_particle():
    total_start_time = time.time()

    # --- PHASE 1: BAKING ---
    baking_start = time.time()
    topo_fn, trans_fn = build_relativistic_kernels(DELTA_POS)

    n_neighbors = len(DELTA_POS)
    MAX_CAPACITY = int(((2 * BOX_RADIUS + 1) ** 2) * n_neighbors * 1.5)

    algebra = FieldAlgebra(dimensions=1, dtype=np.complex128)
    topology = Topology(reachable_func=None, state_class=State)

    gen_data = GenericMarkovianFieldGeneratorData(
        mapper=FieldMapper(algebra, State),
        topology=topology,
        transition_function=trans_fn,
        maximum_step_baking=STEPS,
        max_size=MAX_CAPACITY,
        state_shape=(3,),
        implicit_norm=False,
        explicit_norm=False
    )

    topo_contract = TopologyKernelDataContract(topo_fn, State, MAX_CAPACITY, 3, np.float64)
    topo_cm = TopologyComponentManager.create_from_raw_data(topo_contract, NumbaTopologyStorage(topo_contract),
                                                            NumbaTopologyTranslator(), NumbaTopologyUtility)

    gen_contract = GeneratorKernelDataContract.from_domain(gen_data, global_field_dim=1)
    gen_cm = GeneratorComponentManager(gen_contract, NumbaComplexCSRGeneratorStorage(gen_contract),
                                       GenericGeneratorTranslator(), GenericGeneratorKernelUtility, trans_fn)

    particle_states = np.zeros((NUM_PARTICLES, 3), dtype=np.float64)
    for i in range(NUM_PARTICLES):
        particle_states[i, 0] = float((i - NUM_PARTICLES / 2) * 2)
        particle_states[i, 1] = 0.0
        particle_states[i, 2] = 0.0

    print(f"[*] Baking Bounded Environment (Box Radius {BOX_RADIUS}, Max Capacity {MAX_CAPACITY})...")
    # ONLY the main warmup. No hacks. No clearing internal maps.
    topo_cm.warmup([State(np.array([0.0, 0.0, 0.0]))], steps=BOX_RADIUS)

    u_coords = np.array(topo_cm.fast_refs.handle_map)
    u_weights = np.ones((len(u_coords), 1), dtype=np.complex128)

    for i in range(len(u_coords)):
        x, y, _ = u_coords[i]
        if abs(x) >= BOX_RADIUS - 2 or abs(y) >= BOX_RADIUS - 2:
            u_weights[i, 0] = 0.0 + 0.0j

    field_contract = FieldKernelDataContract(
        max_active_states=MAX_CAPACITY,
        state_dimensions=3,
        field_dimensions=1,
        algebra=algebra,
        state_class_ref=State,
        initial_capacity=MAX_CAPACITY
    )

    global_field_cm = FieldComponentManager.create_from_raw(NumbaComplexUtility, field_contract,
                                                            NumbaComplexFieldKernelStorage(field_contract),
                                                            NumbaFieldTranslator(), u_coords, u_weights)

    gen_cm.inject_environment(topo_cm, global_field_cm)
    baking_time = time.time() - baking_start
    print(f"    [+] Baking completed in {baking_time:.3f} seconds.\n")

    # --- PHASE 2: EXECUTION ---
    print(f"[*] Starting Relativistic Execution Loop ({ITERATIONS} iterations)...")

    # SAFE JIT WARMUP (Generator only, using the standard clear() API)
    print("[*] Warming up engine...")
    gen_cm.clear()
    dummy_p = np.zeros((1, 3))
    gen_cm.load_initial_state(dummy_p, np.ones((1, 1), dtype=np.complex128))
    _ = gen_cm.generate_steps(STEPS)

    _ = sequential_relativistic_collapse(
        np.zeros(3), np.zeros((1, 3)), np.zeros((1, 1), dtype=np.complex128),
        np.zeros((1, 2)), 0, DELTA_POS, 1
    )
    gen_cm.clear()

    exec_start = time.time()

    history_wave = []
    history_particles = []

    for iteration in range(ITERATIONS):
        gen_cm.clear()
        initial_phases = np.ones((NUM_PARTICLES, 1), dtype=np.complex128)
        gen_cm.load_initial_state(particle_states, initial_phases)

        for s in range(STEPS):
            g_states, g_fields = gen_cm.generate_steps(1)
            history_wave.append((g_states.copy(), np.abs(g_fields[:, 0]) ** 2))
            history_particles.append(particle_states.copy())

        claimed_registry = np.zeros((NUM_PARTICLES, 2), dtype=np.float64)
        num_claimed = 0

        for k in range(NUM_PARTICLES):
            chosen_state = sequential_relativistic_collapse(
                particle_states[k], g_states, g_fields, claimed_registry, num_claimed, DELTA_POS, STEPS
            )
            claimed_registry[num_claimed, 0] = chosen_state[0]
            claimed_registry[num_claimed, 1] = chosen_state[1]
            num_claimed += 1
            particle_states[k] = chosen_state

        history_wave.append((np.empty((0, 3)), np.empty(0)))
        history_particles.append(particle_states.copy())

    exec_time = time.time() - exec_start
    print(f"    [+] Physics execution completed in {exec_time:.3f} seconds.\n")

    return history_wave, history_particles, baking_time, exec_time

### ⚡ Execution & Visualization Phase
Here, we inject the environment parameters into the Component Managers and execute the batched temporal loop.

The pipeline executes in three phases:
1. **The Bridge (Baking):** The engine dynamically maps the 3D coordinate space into a continuous, flat 1D array to guarantee L1 CPU cache hits during math execution.
2. **JIT Warmup:** We pass a dummy workload through the engine to force the LLVM compiler to translate our Python functions into machine code.
3. **Physics Math:** The batched C-loop processes multiple particles through the generated quantum wave fields over a set number of iterations.

*(Watch the performance profile output below the video to see the execution time of the compiled math vs. the video rendering).*

In [ ]:
# @title ⚡ 4. Execute Simulation & Display Video
from IPython.display import HTML
from base64 import b64encode
import os

# 1. Run the Engine and get the history data
history_wave, history_particles, b_time, e_time = run_relativistic_multi_particle()

# 2. Render the video!
print("[*] Dispatching to Animator...")
anim_start = time.time()
animate_relativistic_walk(history_wave, history_particles, DELTA_POS, STEPS, ITERATIONS, PATH)
print(f"    [+] Video Render completed in {time.time() - anim_start:.3f} seconds.\n")

# 3. Embed the generated MP4 directly in the Colab output
video_path = os.path.join(PATH, "relativistic_field_analysis.mp4")

if os.path.exists(video_path):
    mp4 = open(video_path, 'rb').read()
    data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
    display(HTML(f"""
    <div style="display: flex; justify-content: center; padding-top: 20px;">
        <video width=800 controls autoplay loop style="border: 1px solid #444; border-radius: 5px;">
              <source src="{data_url}" type="video/mp4">
        </video>
    </div>
    """))
else:
    print("\n❌ Video file not found. Check the output logs above.")